<a href="https://colab.research.google.com/github/shreyagoel0501/shreyagoel0501.github.io/blob/main/Week%203/ex1_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JulesMalin/isba2411-nlp/blob/main/Week%203/ex1_text_classification.ipynb)

# ISBA 2411 - In-Class Exercise 1: The feature face-off

**Supervised Text Classification & Representation - Week 3, Lecture 5**

**Goal (~15 min, pairs).** Build a baseline text classifier, change exactly ONE thing about how the text is turned into features, and measure whether your change actually helped on data the model has never seen.

This is the same shape as **Milestone 2**: a baseline, one deliberate improvement, and a number that says whether it worked.

**Setup.** Pure Python, no API key. Run the warm-up cell once so the runtime is ready, then work top to bottom. One person drives, one person reads the numbers out loud and writes down the answers.

In [1]:
# Warm up the runtime and confirm the packages are present.
import sklearn, pandas as pd
print('scikit-learn', sklearn.__version__)
print('pandas', pd.__version__)

scikit-learn 1.6.1
pandas 2.2.2


---
## 1. Load a small labeled review set

We use **rotten_tomatoes**: short movie-review sentences, each labeled `0` (negative) or `1` (positive). It is small (about 8,500 training reviews, 1,000 test), already split into train and test, and downloads in seconds.

> If the room wifi blocks the download, skip to the **Fallback** section at the bottom, run that one cell, then come back here.

In [2]:
# Load the movie reviews from a CSV bundled in the course repo.
# This never calls Hugging Face, so there are no downloads to fail and no
# rate limits -- it just works, in Colab or anywhere with internet.
import pandas as pd

URL = "https://raw.githubusercontent.com/JulesMalin/isba2411-nlp/main/Week%203/rotten_tomatoes.csv"
try:
    df = pd.read_csv(URL)                 # primary: read straight from GitHub
except Exception:
    df = pd.read_csv("rotten_tomatoes.csv")  # offline fallback: upload the file to Colab

train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]
train_text, train_label = train["text"].tolist(), train["label"].tolist()
test_text,  test_label  = test["text"].tolist(),  test["label"].tolist()

labels = {0: "negative", 1: "positive"}
print("train reviews:", len(train_text), "  test reviews:", len(test_text))
print("\nexample:", labels[train_label[0]], "->", train_text[0])

train reviews: 8530   test reviews: 1066

example: positive -> the rock is destined to be the 21st century's new " conan " and that he's going to make a splash even greater than arnold schwarzenegger , jean-claud van damme or steven segal .


---
## 2. Build the baseline

The recipe is the one from lecture: turn each review into a **TF-IDF** vector, then train a **logistic regression** classifier.

The baseline keeps every knob plain on purpose: unigrams only, no `min_df`, no stopword list. Those are exactly the knobs you will change in Step 3.

Notice we print **two** numbers. The model has already seen the training reviews, so its training accuracy is always flattering. Only the **test** number tells you anything real.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vec = TfidfVectorizer()            # plain baseline: the knobs to change all live here
Xtr = vec.fit_transform(train_text)
Xte = vec.transform(test_text)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xtr, train_label)

train_acc = accuracy_score(train_label, clf.predict(Xtr))
baseline_acc = accuracy_score(test_label, clf.predict(Xte))

print(f'train accuracy: {train_acc:.4f}   (the model already saw this data)')
print(f'TEST  accuracy: {baseline_acc:.4f}   <-- write this number down')
print(f'\nfeatures (vocabulary size): {len(vec.vocabulary_):,}')

train accuracy: 0.8950   (the model already saw this data)
TEST  accuracy: 0.7833   <-- write this number down

features (vocabulary size): 16,474


**Pause and look at the gap.** Training accuracy sits well above test accuracy. A model that scored near-perfect on data it trained on can score far lower on data it has never seen. The lesson for your project: report the **test** number, never the training one.

Baseline test accuracy is your reference point. Everything in the next step is measured against it.

---
## 3. Make ONE deliberate change

Pick **exactly one** knob and change it. Keep everything else identical, so any movement in the score is caused by your one change and nothing else.

Three options, each a different idea about representation:

| Change | What it does | What you might expect |
|---|---|---|
| `ngram_range=(1, 2)` | adds **bigrams**, so word pairs become features | catches phrases like *not good* that unigrams miss |
| `min_df=2` | drops terms that appear in fewer than 2 reviews | removes rare noise, shrinks the feature space |
| `stop_words='english'` | removes very common words (*the*, *a*, *is*) | fewer features, but you might delete real signal |

Uncomment **one** line below, leave the other two commented, then run.

In [8]:
# Change exactly ONE thing. Uncomment ONE line; leave the others commented.

# vec2 = TfidfVectorizer(ngram_range=(1, 2))      # add bigrams
vec2 = TfidfVectorizer(min_df=2)              # drop very rare terms
# vec2 = TfidfVectorizer(stop_words='english')  # remove common words

Xtr2 = vec2.fit_transform(train_text)
Xte2 = vec2.transform(test_text)
clf2 = LogisticRegression(max_iter=1000).fit(Xtr2, train_label)
new_acc = accuracy_score(test_label, clf2.predict(Xte2))

print(f'baseline TEST accuracy : {baseline_acc:.4f}')
print(f'your change TEST acc.  : {new_acc:.4f}')
print(f'change                 : {new_acc - baseline_acc:+.4f}')
print(f'features now            : {len(vec2.vocabulary_):,}  (baseline was {len(vec.vocabulary_):,})')

baseline TEST accuracy : 0.7833
your change TEST acc.  : 0.7767
change                 : -0.0066
features now            : 8,730  (baseline was 16,474)


Did it help, hurt, or do almost nothing? Not every reasonable change improves the test score. That surprise is the point. If you have time, reset the cell and try a different option to compare.

---
## 4. Find a review the model still gets wrong

A single accuracy number hides the individual mistakes. Pull out a few reviews your improved model still gets wrong, and read them. Some are genuinely hard; some are plain errors a person would not make.

In [9]:
pred2 = clf2.predict(Xte2)
wrong = [i for i in range(len(test_label)) if pred2[i] != test_label[i]]
print(f'{len(wrong)} of {len(test_label)} test reviews are still wrong\n')

for i in wrong[:6]:
    print(f"[true {labels[test_label[i]]:>8} | pred {labels[pred2[i]]:>8}]  {test_text[i][:130]}")

238 of 1066 test reviews are still wrong

[true positive | pred negative]  it's like a " big chill " reunion of the baader-meinhof gang , only these guys are more harmless pranksters than political activis
[true positive | pred negative]  weighty and ponderous but every bit as filling as the treat of the title .
[true positive | pred negative]  mostly , [goldbacher] just lets her complicated characters be unruly , confusing and , through it all , human .
[true positive | pred negative]  this is a fascinating film because there is no clear-cut hero and no all-out villain .
[true positive | pred negative]  devotees of star trek ii : the wrath of khan will feel a nagging sense of deja vu , and the grandeur of the best next generation e
[true positive | pred negative]  the main story . . . is compelling enough , but it's difficult to shrug off the annoyance of that chatty fish .


In [10]:
# Now try your own. Write a couple of reviews and see how the model labels them.
my_reviews = [
    'this is not a good film',
    'a sharp, funny movie that earns its ending',
]
for s, p in zip(my_reviews, clf2.predict(vec2.transform(my_reviews))):
    print(f'{labels[p]:>8}  <-  {s}')

positive  <-  this is not a good film
positive  <-  a sharp, funny movie that earns its ending


---
## Bring back an answer to

Be ready to report these out:

1. **Which single change moved accuracy the most, and why?**
2. **Did your "improvement" help on the test set, or only on training?**
3. **What kind of review fools the model every time?** Share one example you found.

**Connection to your final project.** What you just did is Milestone 2 in miniature: a baseline, one deliberate change, and a metric measured on held-out data. Reuse this notebook as the starting scaffold for your own dataset.

---
## Fallback (only if you have no internet at all)

Section 1 already reads the data straight from the course repo on GitHub, so it
normally just works. If the room has **no internet**, download
`rotten_tomatoes.csv` from the repo's `Week 3` folder ahead of time, upload it to
the Colab file panel, and run the cell below instead of Section 1. Everything
after it stays the same.

In [ ]:
import pandas as pd

df = pd.read_csv("rotten_tomatoes.csv")   # uploaded to the Colab file panel
train = df[df["split"] == "train"]
test  = df[df["split"] == "test"]
train_text, train_label = train["text"].tolist(), train["label"].tolist()
test_text,  test_label  = test["text"].tolist(),  test["label"].tolist()

labels = {0: "negative", 1: "positive"}
print("train reviews:", len(train_text), "  test reviews:", len(test_text))

---
### Reading connection

| Idea tonight | Where to read more |
|---|---|
| Text classification, sentiment, evaluation | J&M ch. 4 |
| Bag of words, TF-IDF, the term-document matrix | J&M ch. 4; Tunstall ch. 2 |
| Classification with the Hugging Face toolkit | HOLLM ch. 4 |

*ISBA 2411 - Natural Language Processing & AI - Summer 2026*